# 🤖 Multi-Agent Governance
**Agent Governance Toolkit — Interactive Demo**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microsoft/agent-governance-toolkit/blob/main/notebooks/03_multi_agent_governance.ipynb)

In this notebook you will:
- Set up SLOs (Service Level Objectives) for multiple agents
- Simulate a circuit breaker tripping under load
- Run a basic chaos test across an agent fleet
- View per-agent health dashboards

> **No API key required** — this demo runs fully offline.

## Step 1 — Install the toolkit

In [ ]:
!pip install agent-governance-toolkit[full] -q

## Step 2 — Define Agent SLOs

In [ ]:
import random
import time
from dataclasses import dataclass, field
from datetime import datetime
from typing import List

@dataclass
class AgentSLO:
    """Service Level Objective for a single agent."""
    agent_id: str
    max_error_rate: float = 0.05   # 5% max errors
    max_latency_ms: float = 2000.0  # 2s max latency
    min_availability: float = 0.99  # 99% uptime

@dataclass
class AgentMetrics:
    """Live metrics for an agent."""
    agent_id: str
    total_calls: int = 0
    errors: int = 0
    total_latency_ms: float = 0.0
    circuit_open: bool = False
    history: List[dict] = field(default_factory=list)

    @property
    def error_rate(self):
        return self.errors / self.total_calls if self.total_calls else 0.0

    @property
    def avg_latency_ms(self):
        return self.total_latency_ms / self.total_calls if self.total_calls else 0.0

# Define a fleet of 3 agents with SLOs
agents = {
    "research-agent":  AgentSLO("research-agent",  max_error_rate=0.05),
    "writing-agent":   AgentSLO("writing-agent",   max_error_rate=0.03),
    "data-agent":      AgentSLO("data-agent",       max_error_rate=0.10),
}

metrics = {aid: AgentMetrics(aid) for aid in agents}

print("Agent fleet configured:")
for aid, slo in agents.items():
    print(f"  {aid:<20} max_error_rate={slo.max_error_rate:.0%}  max_latency={slo.max_latency_ms:.0f}ms")

## Step 3 — Circuit Breaker Implementation

In [ ]:
def check_circuit_breaker(agent_id: str, slo: AgentSLO, m: AgentMetrics) -> bool:
    """Open circuit if SLO is breached. Returns True if circuit is CLOSED (healthy)."""
    if m.total_calls < 5:
        return True  # not enough data yet
    if m.error_rate > slo.max_error_rate:
        m.circuit_open = True
        print(f"  ⚡ Circuit OPEN for '{agent_id}' — error rate {m.error_rate:.1%} > SLO {slo.max_error_rate:.1%}")
        return False
    if m.avg_latency_ms > slo.max_latency_ms:
        m.circuit_open = True
        print(f"  ⚡ Circuit OPEN for '{agent_id}' — latency {m.avg_latency_ms:.0f}ms > SLO {slo.max_latency_ms:.0f}ms")
        return False
    m.circuit_open = False
    return True

def simulate_call(agent_id: str, slo: AgentSLO, m: AgentMetrics,
                  error_rate: float = 0.05, latency_range=(100, 500)):
    """Simulate a single agent call and update metrics."""
    if m.circuit_open:
        return {"status": "CIRCUIT_OPEN", "agent": agent_id}
    latency = random.uniform(*latency_range)
    is_error = random.random() < error_rate
    m.total_calls += 1
    m.total_latency_ms += latency
    if is_error:
        m.errors += 1
    check_circuit_breaker(agent_id, slo, m)
    return {"status": "ERROR" if is_error else "OK", "latency_ms": latency}

print("Circuit breaker ready.")

## Step 4 — Simulate Normal Load

In [ ]:
random.seed(42)
print("Simulating 20 calls per agent under normal load...\n")

for _ in range(20):
    simulate_call("research-agent", agents["research-agent"], metrics["research-agent"], error_rate=0.03)
    simulate_call("writing-agent",  agents["writing-agent"],  metrics["writing-agent"],  error_rate=0.02)
    simulate_call("data-agent",     agents["data-agent"],     metrics["data-agent"],     error_rate=0.08)

print(f"{'Agent':<20} {'Calls':<8} {'Errors':<8} {'Error Rate':<12} {'Avg Latency':<14} {'Circuit'}")
print("-" * 75)
for aid, m in metrics.items():
    slo_status = "🔴 OPEN" if m.circuit_open else "🟢 CLOSED"
    print(f"{aid:<20} {m.total_calls:<8} {m.errors:<8} {m.error_rate:<12.1%} {m.avg_latency_ms:<14.0f} {slo_status}")

## Step 5 — Chaos Test: Inject High Error Rate

In [ ]:
print("🔥 Chaos test: injecting 50% error rate into 'research-agent'...\n")

for i in range(10):
    result = simulate_call(
        "research-agent",
        agents["research-agent"],
        metrics["research-agent"],
        error_rate=0.50,  # chaos: 50% errors
    )
    if result["status"] == "CIRCUIT_OPEN":
        print(f"  Call {i+1}: Rejected — circuit is open")
    else:
        print(f"  Call {i+1}: {result['status']}  latency={result['latency_ms']:.0f}ms")

print()
print(f"{'Agent':<20} {'Calls':<8} {'Error Rate':<12} {'Circuit'}")
print("-" * 55)
for aid, m in metrics.items():
    slo_status = "🔴 OPEN" if m.circuit_open else "🟢 CLOSED"
    print(f"{aid:<20} {m.total_calls:<8} {m.error_rate:<12.1%} {slo_status}")

## Step 6 — Fleet Health Dashboard

In [ ]:
print("\n" + "═" * 60)
print("  AGENT FLEET HEALTH DASHBOARD")
print("  Generated:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("═" * 60)

for aid, m in metrics.items():
    slo = agents[aid]
    er_ok  = m.error_rate  <= slo.max_error_rate
    lat_ok = m.avg_latency_ms <= slo.max_latency_ms
    health = "HEALTHY" if (er_ok and lat_ok and not m.circuit_open) else "DEGRADED"
    icon   = "✅" if health == "HEALTHY" else "⚠️ "

    print(f"\n  {icon} {aid.upper()}  [{health}]")
    print(f"     Total calls : {m.total_calls}")
    print(f"     Error rate  : {m.error_rate:.1%}  (SLO: ≤{slo.max_error_rate:.0%})  {'✅' if er_ok else '❌'}")
    print(f"     Avg latency : {m.avg_latency_ms:.0f}ms  (SLO: ≤{slo.max_latency_ms:.0f}ms)  {'✅' if lat_ok else '❌'}")
    print(f"     Circuit     : {'🔴 OPEN' if m.circuit_open else '🟢 CLOSED'}")

print("\n" + "═" * 60)

## ✅ What You Learned

- How to define SLOs for individual agents in a fleet
- How circuit breakers protect downstream systems from degraded agents
- How chaos testing reveals SLO breaches before they hit production
- How to build a real-time fleet health dashboard

---

**Explore more:**
- [Policy Enforcement 101 →](./01_policy_enforcement_101.ipynb)
- [MCP Security Proxy →](./02_mcp_security_proxy.ipynb)
- [Full documentation →](https://github.com/microsoft/agent-governance-toolkit)